# ETL Process and Data Pipeline

ETL (Extract, Transform, Load) is a data processing pipeline used for integrating data from various sources. This notebook covers how to implement an ETL process and create data pipelines using Python.

---

### Run this first: three sources to build a pipeline from

A real ETL job pulls from more than one place. This cell gives you all three,
locally, with no network and no credentials:

1. **A CSV drop zone** with daily files, some of them messy
2. **A source database** (SQLite) with customers and orders
3. **A small API** running inside this notebook, serving paginated JSON

It also creates `example.csv` and a `df`, so the cells further down this notebook
have something real to work on.

In [ ]:
import os, sqlite3, json, random, threading, time
import pandas as pd
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

# ---------------------------------------------------------------- 1. CSV drop zone
os.makedirs("landing", exist_ok=True)

DAILY = {
    "2026-05-01": [
        "5001,101,Laptop Pro,1,450000,NG,completed",
        "5002,102,Phone X,2,180000,GH,completed",
        "5003,101,Wireless Mouse,3,8000,NG,completed",
    ],
    "2026-05-02": [
        "5004,103,Laptop Pro,1,520000,NG,pending",
        "5005,104,Monitor 27,2,95000,KE,completed",
        "5006,102,Wireless Mouse,1,N/A,GH,cancelled",      # unparseable price
    ],
    "2026-05-03": [
        "5007,103,Keyboard,2,15000,NG,completed",
        "5008,,Phone X,1,180000,KE,completed",             # missing customer
        "5009,999,Headset,4,22000,GH,completed",           # customer does not exist
        "5007,103,Keyboard,2,15000,NG,completed",          # duplicate of 5007
    ],
}

for day, rows in DAILY.items():
    with open(f"landing/orders_{day}.csv", "w", encoding="utf-8") as f:
        f.write("order_id,customer_id,item,quantity,unit_price,country,status\n")
        for row in rows:
            f.write(row + "\n")

# ---------------------------------------------------------------- 2. source database
if os.path.exists("source.db"):
    os.remove("source.db")

source = sqlite3.connect("source.db")
source.executescript('''
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY, name TEXT, segment TEXT,
    country TEXT, updated_at TEXT);
''')
source.executemany("INSERT INTO customers VALUES (?, ?, ?, ?, ?)", [
    (101, "Ahmed Oladapo", "enterprise", "NG", "2026-04-01T09:00:00"),
    (102, "Lahla Mensah",  "smb",        "GH", "2026-04-02T09:00:00"),
    (103, "Judith Eze",    "enterprise", "NG", "2026-04-03T09:00:00"),
    (104, "Uche Okafor",   "smb",        "KE", "2026-05-01T09:00:00"),
    (105, "Grace Wanjiru", "smb",        "KE", "2026-05-02T09:00:00"),
])
source.commit()
source.close()

# ---------------------------------------------------------------- 3. a small API
PRODUCTS = [
    {"sku": f"SKU-{i:03d}",
     "name": name,
     "category": category,
     "cost": cost,
     "supplier": {"name": supplier, "country": country}}
    for i, (name, category, cost, supplier, country) in enumerate([
        ("Laptop Pro", "Electronics", 380000, "TechDist", "NG"),
        ("Phone X", "Electronics", 140000, "TechDist", "NG"),
        ("Wireless Mouse", "Accessories", 5200, "GadgetHub", "GH"),
        ("Monitor 27", "Electronics", 71000, "TechDist", "NG"),
        ("Keyboard", "Accessories", 9800, "GadgetHub", "GH"),
        ("Headset", "Accessories", 15400, "SoundCo", "KE"),
        ("Webcam", "Accessories", 24000, "SoundCo", "KE"),
    ], start=1)
]
PAGE_SIZE = 3


class Handler(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass

    def do_GET(self):
        parsed = urlparse(self.path)
        if parsed.path != "/api/products":
            body = json.dumps({"error": "not found"}).encode()
            self.send_response(404)
        else:
            page = int(parse_qs(parsed.query).get("page", ["1"])[0])
            pages = max(1, -(-len(PRODUCTS) // PAGE_SIZE))
            start = (page - 1) * PAGE_SIZE
            body = json.dumps({
                "info": {"count": len(PRODUCTS), "pages": pages,
                         "next": f"{API_URL}/api/products?page={page+1}"
                                 if page < pages else None},
                "results": PRODUCTS[start:start + PAGE_SIZE],
            }).encode()
            self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)


class QuietServer(ThreadingHTTPServer):
    daemon_threads = True

    def handle_error(self, request, client_address):
        pass


if "_api" in dir() and _api:
    _api.shutdown()

_api = QuietServer(("127.0.0.1", 0), Handler)
threading.Thread(target=_api.serve_forever, daemon=True).start()
API_URL = f"http://127.0.0.1:{_api.server_address[1]}"
time.sleep(0.3)

# ------------------------------------------- 4. example.csv, for the cells below this
pd.DataFrame({
    "column_name": ["10.5", "20.0", "30.5", None, "50.0"],
    "column1": [1, 2, 3, 4, 5],
    "column2": [10, 20, 30, 40, 50],
}).to_csv("example.csv", index=False)

df = pd.read_csv("example.csv")

print("sources ready")
print(f"  landing/       {len(os.listdir('landing'))} daily CSV file(s)")
print(f"  source.db      customers table")
print(f"  {API_URL}/api/products   paginated JSON")
print(f"  example.csv    {df.shape[0]} rows, so the cells below have data")

## 1. Extracting Data
Data can be extracted from various sources such as CSV files, APIs, and databases.

In [5]:
# # Extract data from a CSV file
# import pandas as pd

# # Read CSV into a pandas DataFrame
# df = pd.read_csv('example.csv')
# print(df.head())

# Extract data from an API
import requests
response = requests.get('https://rickandmortyapi.com/api/character')
if response.status_code == 200:
    api_data = response.json()['info']
    print(api_data)
else:
    print('API request failed')


{'count': 826, 'pages': 42, 'next': 'https://rickandmortyapi.com/api/character?page=2', 'prev': None}


---

## Practice: Extract

### The one rule of extraction

**Land the raw data first, exactly as it arrived, then transform a copy.**

If you clean while you extract, and a week later someone asks why a number looks
wrong, you have no way to answer. Keep the raw file. Storage is cheap, re-extracting
from a source that has since changed is often impossible.

In [ ]:
import pandas as pd, glob, os

# Extracting from files: read every daily drop, stamped with where it came from
def extract_files(pattern="landing/orders_*.csv"):
    """Read every matching CSV, tagging each row with its source file."""
    frames = []
    for path in sorted(glob.glob(pattern)):
        frame = pd.read_csv(path, dtype=str, encoding="utf-8")   # everything as text
        frame["_source_file"] = os.path.basename(path)
        frames.append(frame)
        print(f"  {path}: {len(frame)} row(s)")

    if not frames:
        raise FileNotFoundError(f"no files matched {pattern!r}")
    return pd.concat(frames, ignore_index=True)


raw_orders = extract_files()
print(f"\n{len(raw_orders)} raw row(s) from {raw_orders['_source_file'].nunique()} file(s)")
raw_orders.head()

`dtype=str` looks odd and is deliberate. At extract time you do not want pandas
guessing types, because it guesses differently depending on which rows it sees.
A column that is integers today and has one `"N/A"` tomorrow silently changes dtype,
and your downstream code breaks in a way that is very hard to trace.

Read everything as text, land it, and convert **during transform**, where you control
what happens to the values that will not convert.

In [ ]:
# Extracting from a database, incrementally
import sqlite3

def extract_db(table, since=None, db="source.db"):
    """Read a table, optionally only rows updated since a watermark."""
    connection = sqlite3.connect(db)
    try:
        if since:
            df = pd.read_sql(
                f"SELECT * FROM {table} WHERE updated_at > ? ORDER BY updated_at",
                connection, params=(since,))
        else:
            df = pd.read_sql(f"SELECT * FROM {table}", connection)
    finally:
        connection.close()
    print(f"  {table}: {len(df)} row(s)" + (f" since {since}" if since else " (full)"))
    return df


customers_full = extract_db("customers")
customers_recent = extract_db("customers", since="2026-04-30T00:00:00")
customers_recent

In [ ]:
# Extracting from an API, following the pages
import requests

def extract_api(start_url, max_pages=20):
    """Page through an API, collecting every record."""
    records, pages, url = [], 0, start_url
    while url and pages < max_pages:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        payload = response.json()
        records.extend(payload["results"])
        pages += 1
        url = payload["info"]["next"]
    print(f"  api: {len(records)} record(s) over {pages} page(s)")
    return records


products_raw = extract_api(f"{API_URL}/api/products")
print(products_raw[0])

The cell further up this notebook that calls `rickandmortyapi.com` needs internet.
This one does not, because the API is running on your own machine. The `requests`
code is identical either way, which is the point.

In [ ]:
# Landing the raw extract before touching it
from datetime import date
import json

RUN_DATE = date.today().isoformat()
raw_dir = os.path.join("raw", f"date={RUN_DATE}")
os.makedirs(raw_dir, exist_ok=True)

raw_orders.to_csv(os.path.join(raw_dir, "orders.csv"), index=False)
customers_full.to_csv(os.path.join(raw_dir, "customers.csv"), index=False)
with open(os.path.join(raw_dir, "products.json"), "w", encoding="utf-8") as f:
    json.dump(products_raw, f, indent=2)

for path in sorted(glob.glob("raw/**/*", recursive=True)):
    if os.path.isfile(path):
        print(f"  {path}  ({os.path.getsize(path)} bytes)")

`date=2026-09-14` as a folder name is Hive-style partitioning, the same layout the
pandas notebook used. Every warehouse engine understands it, and it means you can
always go back to exactly what arrived on a given day.

In [ ]:
# An extraction manifest: what did we pull, and does it look normal?
def build_manifest(sources):
    """Record row counts and columns for every source in this run."""
    manifest = {"run_date": RUN_DATE, "sources": {}}
    for name, frame in sources.items():
        manifest["sources"][name] = {
            "rows": len(frame),
            "columns": list(frame.columns),
            "null_counts": {c: int(frame[c].isna().sum()) for c in frame.columns},
        }
    return manifest


manifest = build_manifest({
    "orders": raw_orders,
    "customers": customers_full,
    "products": pd.json_normalize(products_raw),
})

with open(os.path.join(raw_dir, "_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

for name, info in manifest["sources"].items():
    print(f"{name}: {info['rows']} rows, {len(info['columns'])} columns")

Yesterday's manifest is what lets you say "we normally get 40,000 orders and today
we got 4,000" **before** anyone builds a report on the wrong number. A row count
that drops by 90% overnight is nearly always a broken extract, not a quiet week.

### Your turn: tasks

**Task 1.** Write `extract_files(pattern)` that reads every matching CSV with
`dtype=str`, adds a `_source_file` column and returns one DataFrame. Print the row
count per file.

**Task 2.** Extract the `customers` table from `source.db` in full, then again using a
watermark of `"2026-04-30T00:00:00"`, and print both counts.

**Task 3.** Extract every product from the API by following the `next` link, and print
how many pages it took.

**DE Task 4.** Land all three extracts into `raw/date=<today>/` and list the files
with their sizes.

**DE Task 5.** Write `build_manifest(sources)` recording row count, columns and null
counts per source, and save it as `_manifest.json` next to the raw files.

**DE Task 6.** Write `check_volume(manifest, previous, tolerance=0.5)` that compares
this run's row counts against a previous manifest and prints a warning for any source
that moved by more than the tolerance. Test it with a fake previous manifest.

In [ ]:
# Task 1 answer


In [ ]:
# Task 2 answer


In [ ]:
# Task 3 answer


In [ ]:
# DE Task 4 answer


In [ ]:
# DE Task 5 answer


In [ ]:
# DE Task 6 answer


## 2. Transforming Data
Data transformation involves cleaning, normalizing, and restructuring data to fit the desired format.

In [6]:
# Example: Data cleaning and transformation
# Drop missing values and convert column data types
df_cleaned = df.dropna()
df_cleaned['column_name'] = df_cleaned['column_name'].astype(float)

# Data transformation: creating new features
df_cleaned['new_feature'] = df_cleaned['column1'] + df_cleaned['column2']
print(df_cleaned.head())


NameError: name 'df' is not defined

---

## Practice: Transform

Transformation is not one step, it is five, and doing them in this order saves a lot
of pain:

1. **Standardise** the shape: column names, whitespace, casing
2. **Type** the values, coercing what will not convert rather than crashing
3. **Deduplicate** on the business key
4. **Validate**, splitting good rows from bad with a reason for each
5. **Derive** the new columns your users actually asked for

Validation comes before derivation. There is no point computing revenue on a row you
are about to throw away.

In [ ]:
import pandas as pd, numpy as np

# 1. Standardise
def standardise(df):
    """Normalise column names and strip stray whitespace from text columns."""
    out = df.copy()
    out.columns = (out.columns.str.strip().str.lower()
                   .str.replace(" ", "_", regex=False))
    for column in out.columns:
        if not pd.api.types.is_numeric_dtype(out[column]):
            out[column] = (out[column].astype(str).str.strip()
                           .replace({"nan": None, "None": None, "": None}))
    return out


standardised = standardise(raw_orders)
print(list(standardised.columns))
standardised.head(3)

In [ ]:
# 2. Type, and report every coercion
def coerce_types(df):
    """Convert columns to their real types, reporting what would not convert."""
    out = df.copy()
    report = {}

    for column, converter in [
        ("order_id", lambda s: pd.to_numeric(s, errors="coerce")),
        ("customer_id", lambda s: pd.to_numeric(s, errors="coerce")),
        ("quantity", lambda s: pd.to_numeric(s, errors="coerce")),
        ("unit_price", lambda s: pd.to_numeric(s, errors="coerce")),
    ]:
        before = out[column].isna().sum()
        out[column] = converter(out[column])
        after = out[column].isna().sum()
        report[column] = int(after - before)

    for column, coerced in report.items():
        if coerced:
            print(f"  {column}: {coerced} value(s) could not be converted")
    return out, report


typed, coercion_report = coerce_types(standardised)
print(typed.dtypes)

Printing the coercion count is the whole discipline. `errors="coerce"` is only safe
because you say out loud how much it swallowed. A pipeline that coerces 40% of a
column to null without mentioning it is worse than one that crashes.

In [ ]:
# 3. Deduplicate on the business key
def deduplicate(df, key="order_id"):
    """Drop repeats of the business key, returning (kept, dropped)."""
    duplicated = df[df.duplicated(subset=[key], keep="first")].copy()
    kept = df.drop_duplicates(subset=[key], keep="first")
    if len(duplicated):
        print(f"  dropped {len(duplicated)} duplicate {key}(s): "
              f"{sorted(duplicated[key].dropna().unique().tolist())}")
    return kept, duplicated


deduped, dupes = deduplicate(typed)
print(f"{len(typed)} -> {len(deduped)}")

Note it is `drop_duplicates(subset=["order_id"])`, not a bare `drop_duplicates()`.
A bare one only removes rows that are identical in **every** column, so two records
for the same order that differ in a timestamp both survive. Dedupe on the key that
identifies the thing, not on the whole row.

In [ ]:
# 4. Validate, keeping the reason
VALID_STATUSES = {"completed", "pending", "cancelled"}


def validate(df, known_customers):
    """Split into (clean, rejected). Every rejected row carries a reason."""
    reasons = []
    for _, row in df.iterrows():
        if pd.isna(row["order_id"]):
            reasons.append("missing order_id")
        elif pd.isna(row["customer_id"]):
            reasons.append("missing customer_id")
        elif row["customer_id"] not in known_customers:
            reasons.append(f"unknown customer {int(row['customer_id'])}")
        elif pd.isna(row["unit_price"]):
            reasons.append("unit_price could not be parsed")
        elif pd.isna(row["quantity"]) or row["quantity"] < 1:
            reasons.append(f"bad quantity {row['quantity']}")
        elif row["status"] not in VALID_STATUSES:
            reasons.append(f"unexpected status {row['status']!r}")
        else:
            reasons.append(None)

    tagged = df.assign(_reason=reasons)
    clean = tagged[tagged["_reason"].isna()].drop(columns=["_reason"]).copy()
    rejected = tagged[tagged["_reason"].notna()].copy()
    return clean, rejected


known = set(customers_full["customer_id"])
clean, rejected = validate(deduped, known)

print(f"clean {len(clean)}, rejected {len(rejected)}")
for _, row in rejected.iterrows():
    order_id = row["order_id"]
    label = int(order_id) if pd.notna(order_id) else "?"
    print(f"  order {label}: {row['_reason']}")

The "unknown customer 999" rejection is **referential integrity**: the order points
at a customer that does not exist. In a database a foreign key would have prevented
it. Coming from a CSV, nothing stops it, so you check it yourself.

This is one of the most valuable checks in any pipeline, and one of the most commonly
skipped.

In [ ]:
# 5. Derive
def enrich(df, customers):
    """Add the derived columns and join the customer attributes."""
    out = df.merge(
        customers[["customer_id", "name", "segment"]],
        on="customer_id", how="left", validate="many_to_one",
    )
    return out.assign(
        revenue=lambda d: d["quantity"] * d["unit_price"],
        vat=lambda d: d["revenue"] * 0.075,
        total=lambda d: d["revenue"] + d["vat"],
        order_date=lambda d: d["_source_file"].str.extract(r"(\d{4}-\d{2}-\d{2})")[0],
        value_band=lambda d: np.select(
            [d["revenue"] > 400000, d["revenue"] > 50000],
            ["large", "medium"], default="small"),
    )


enriched = enrich(clean, customers_full)
enriched[["order_id", "name", "segment", "revenue", "total", "order_date", "value_band"]]

`validate="many_to_one"` on the merge again, because a duplicated customer row would
silently multiply your orders and inflate revenue. It costs nothing and it has saved
more pipelines than any other single argument in pandas.

### Data quality gates: when to stop the run

Checks are only useful if something happens when they fail. Decide, per check,
whether it is a **warning** (log it, carry on) or a **gate** (stop the run).

In [ ]:
def quality_gates(clean, rejected, coercion_report, max_reject_rate=0.40):
    """Return (passed, [messages]). A failed gate should stop the load."""
    messages = []
    passed = True
    total = len(clean) + len(rejected)

    rate = len(rejected) / total if total else 0
    if rate > max_reject_rate:
        passed = False
        messages.append(f"GATE FAILED: {rate:.0%} rejected, limit is {max_reject_rate:.0%}")
    else:
        messages.append(f"ok: rejection rate {rate:.0%}")

    if clean.empty:
        passed = False
        messages.append("GATE FAILED: no clean rows at all")

    if clean["order_id"].duplicated().any():
        passed = False
        messages.append("GATE FAILED: duplicate order_id survived deduplication")

    negative = (clean["revenue"] < 0).sum() if "revenue" in clean else 0
    if negative:
        passed = False
        messages.append(f"GATE FAILED: {negative} row(s) with negative revenue")

    for column, coerced in coercion_report.items():
        if coerced:
            messages.append(f"warning: {coerced} value(s) coerced in {column}")

    return passed, messages


passed, messages = quality_gates(enriched, rejected, coercion_report)
for message in messages:
    print(message)
print()
print("proceed to load" if passed else "STOP, do not load")

### Your turn: tasks

**Task 1.** Write `standardise(df)` that lowercases and underscores the column names
and strips whitespace from text columns.

**Task 2.** Write `coerce_types(df)` that converts the numeric columns with
`errors="coerce"` and returns both the DataFrame and a report of how many values it
turned into nulls.

**Task 3.** Write `deduplicate(df, key)` that returns `(kept, dropped)` and prints the
duplicated keys.

**DE Task 4.** Write `validate(df, known_customers)` returning `(clean, rejected)`
with a reason column. Include the referential integrity check against the customer
list.

**DE Task 5.** Write `enrich(df, customers)` that joins the customer attributes with
`validate="many_to_one"` and adds revenue, vat, total and a value band.

**DE Task 6.** Write `quality_gates(...)` with at least four checks, returning
`(passed, messages)`. Make one of them fail deliberately by lowering the threshold,
and show the run being stopped.

In [ ]:
# Task 1 answer


In [ ]:
# Task 2 answer


In [ ]:
# Task 3 answer


In [ ]:
# DE Task 4 answer


In [ ]:
# DE Task 5 answer


In [ ]:
# DE Task 6 answer


## 3. Loading Data
Once the data is transformed, it can be loaded into a database or another storage system.

In [ ]:
# Load the cleaned data into a new CSV file
df_cleaned.to_csv('transformed_data.csv', index=False)

# Load data into a SQL database
import sqlite3
conn = sqlite3.connect('example.db')
df_cleaned.to_sql('transformed_table', conn, if_exists='replace', index=False)
conn.close()


---

## Practice: Load

### The load must be safe to run twice

Every job gets re-run: a retry after a timeout, a backfill, someone clicking the
button again because they were not sure it worked. A load that creates duplicates on
the second run is not finished.

In [ ]:
import sqlite3, os
from contextlib import closing
from datetime import datetime

if os.path.exists("warehouse.db"):
    os.remove("warehouse.db")

with closing(sqlite3.connect("warehouse.db")) as wh:
    wh.executescript('''
    CREATE TABLE fact_orders (
        order_id    INTEGER PRIMARY KEY,
        customer_id INTEGER,
        customer_name TEXT,
        segment     TEXT,
        item        TEXT,
        quantity    INTEGER,
        unit_price  REAL,
        revenue     REAL,
        vat         REAL,
        total       REAL,
        country     TEXT,
        status      TEXT,
        order_date  TEXT,
        value_band  TEXT,
        source_file TEXT,
        loaded_at   TEXT
    );

    CREATE TABLE rejected_orders (
        order_id TEXT, reason TEXT, source_file TEXT, rejected_at TEXT);

    CREATE TABLE etl_runs (
        run_id      INTEGER PRIMARY KEY AUTOINCREMENT,
        pipeline    TEXT,
        started_at  TEXT,
        finished_at TEXT,
        rows_read   INTEGER,
        rows_loaded INTEGER,
        rows_rejected INTEGER,
        status      TEXT,
        message     TEXT
    );

    CREATE TABLE etl_watermark (
        source TEXT PRIMARY KEY, last_value TEXT, updated_at TEXT);
    ''')
    wh.commit()

print("warehouse.db created")

In [ ]:
FACT_COLUMNS = [
    "order_id", "customer_id", "customer_name", "segment", "item", "quantity",
    "unit_price", "revenue", "vat", "total", "country", "status", "order_date",
    "value_band", "source_file", "loaded_at",
]

UPSERT = f'''
INSERT INTO fact_orders ({", ".join(FACT_COLUMNS)})
VALUES ({", ".join("?" for _ in FACT_COLUMNS)})
ON CONFLICT(order_id) DO UPDATE SET
{", ".join(f"{c} = excluded.{c}" for c in FACT_COLUMNS if c != "order_id")}
'''


def load_fact(df, db="warehouse.db"):
    """Upsert rows into fact_orders inside one transaction. Idempotent."""
    out = df.copy()
    out["customer_name"] = out["name"]
    out["source_file"] = out["_source_file"]
    out["loaded_at"] = datetime.now().isoformat(timespec="seconds")

    payload = [tuple(None if pd.isna(v) else v for v in row)
               for row in out[FACT_COLUMNS].itertuples(index=False, name=None)]

    with closing(sqlite3.connect(db)) as wh:
        with wh:
            wh.executemany(UPSERT, payload)
        return wh.execute("SELECT COUNT(*) FROM fact_orders").fetchone()[0]


import pandas as pd

for attempt in [1, 2, 3]:
    total = load_fact(enriched)
    print(f"run {attempt}: {len(enriched)} rows loaded, {total} in the warehouse")

Three runs, same total. That is idempotency, and it is the property that lets you
schedule a job without fear.

`itertuples(index=False, name=None)` is the fast way to turn a DataFrame into tuples
for `executemany`. The `None if pd.isna(v)` wrapper matters too: pandas uses `NaN`
for missing numbers, and SQLite would happily store the float `nan` rather than a
proper `NULL`.

In [ ]:
# Verify after loading. A pipeline that does not check its own output is a guess.
def verify_load(df, db="warehouse.db"):
    """Compare what we meant to load against what is actually in the warehouse."""
    expected_rows = len(df)
    expected_revenue = float(df["revenue"].sum())

    with closing(sqlite3.connect(db)) as wh:
        actual_rows, actual_revenue = wh.execute(
            "SELECT COUNT(*), COALESCE(SUM(revenue), 0) FROM fact_orders").fetchone()

    rows_ok = actual_rows == expected_rows
    revenue_ok = abs(actual_revenue - expected_revenue) < 0.01

    print(f"rows:    expected {expected_rows}, found {actual_rows}   {'ok' if rows_ok else 'MISMATCH'}")
    print(f"revenue: expected {expected_revenue:,.2f}, found {actual_revenue:,.2f}   "
          f"{'ok' if revenue_ok else 'MISMATCH'}")
    return rows_ok and revenue_ok


verify_load(enriched)

In [ ]:
# Loading the rejects too, so nothing disappears silently
def load_rejects(rejected, db="warehouse.db"):
    """Store rejected rows with their reason, for someone to investigate."""
    if rejected.empty:
        return 0
    now = datetime.now().isoformat(timespec="seconds")
    payload = [
        (str(row["order_id"]), row["_reason"], row["_source_file"], now)
        for _, row in rejected.iterrows()
    ]
    with closing(sqlite3.connect(db)) as wh:
        with wh:
            wh.execute("DELETE FROM rejected_orders")          # this run replaces the last
            wh.executemany(
                "INSERT INTO rejected_orders VALUES (?, ?, ?, ?)", payload)
        return wh.execute("SELECT COUNT(*) FROM rejected_orders").fetchone()[0]


print(f"{load_rejects(rejected)} rejected row(s) stored for review")

with closing(sqlite3.connect("warehouse.db")) as wh:
    for row in wh.execute("SELECT order_id, reason, source_file FROM rejected_orders"):
        print(f"  {row}")

A rejects table is what turns "we dropped some rows" into a question somebody can
answer. Without it, the only evidence a row ever existed is that the totals do not
add up.

In [ ]:
# Full refresh versus incremental, and when each is right
def load_full_refresh(df, table, db="warehouse.db"):
    """Replace the whole table. Simple, safe, only viable while the table is small."""
    with closing(sqlite3.connect(db)) as wh:
        with wh:
            deleted = wh.execute(f"DELETE FROM {table}").rowcount
            # ... then insert everything
        return deleted


print("Full refresh:")
print("  + dead simple, always consistent, no watermark to get wrong")
print("  + a fix to old data is picked up automatically")
print("  - re-reads everything every run. hopeless past a few million rows")
print()
print("Incremental:")
print("  + only moves what changed. scales")
print("  - needs a reliable updated_at, and deletes are invisible")
print("  - a late-arriving row with an old timestamp is missed forever")
print()
print("The usual answer: incremental daily, full refresh weekly to heal the drift.")

In [ ]:
# Writing partitioned files alongside the database load
def load_files(df, base="warehouse_files"):
    """Write one file per order_date partition."""
    written = []
    for partition, group in df.dropna(subset=["order_date"]).groupby("order_date"):
        folder = os.path.join(base, f"date={partition}")
        os.makedirs(folder, exist_ok=True)
        path = os.path.join(folder, "orders.csv")
        group.to_csv(path, index=False)
        written.append((path, len(group)))
    return written


for path, rows in load_files(enriched):
    print(f"  {path}  ({rows} rows)")

### Your turn: tasks

**Task 1.** Create `warehouse.db` with a `fact_orders` table that has `order_id` as
its primary key.

**Task 2.** Write `load_fact(df)` using `ON CONFLICT(order_id) DO UPDATE`, and run it
three times, showing the row count does not change.

**Task 3.** Write `verify_load(df)` comparing the expected and actual row count and
revenue total.

**DE Task 4.** Write `load_rejects(rejected)` storing every rejected row with its
reason, and print the table afterwards.

**DE Task 5.** Change one order's status in the source file, re-run the whole
transform and load, and prove the warehouse row was **updated** rather than a second
row being inserted.

**DE Task 6.** Write `load_files(df)` producing `warehouse_files/date=YYYY-MM-DD/`
partitions, then read them all back and confirm the total revenue matches the
database.

In [ ]:
# Task 1 answer


In [ ]:
# Task 2 answer


In [ ]:
# Task 3 answer


In [ ]:
# DE Task 4 answer


In [ ]:
# DE Task 5 answer


In [ ]:
# DE Task 6 answer


## 4. Building Data Pipelines
A data pipeline automates the ETL process, ensuring that data flows from source to destination on a regular schedule.

In [ ]:
# Example of a simple data pipeline in Python
def extract():
    df = pd.read_csv('example.csv')
    return df

def transform(df):
    df_cleaned = df.dropna()
    df_cleaned['new_feature'] = df_cleaned['column1'] + df_cleaned['column2']
    return df_cleaned

def load(df):
    df.to_csv('final_output.csv', index=False)

# Run the pipeline
data = extract()
transformed_data = transform(data)
load(transformed_data)


---

## Practice: turning scripts into a pipeline

### Stop using `print`

`print` goes to the screen and disappears. A scheduled job runs at 3am with nobody
watching, so you need output with a timestamp, a level and a destination.

In [ ]:
import logging, sys

logger = logging.getLogger("dec_pipeline")
logger.setLevel(logging.INFO)
logger.handlers.clear()                       # safe to re-run this cell

formatter = logging.Formatter(
    "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

console = logging.StreamHandler(sys.stdout)
console.setFormatter(formatter)
logger.addHandler(console)

file_handler = logging.FileHandler("pipeline.log", encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

logger.debug("not shown, the level is INFO")
logger.info("extract started")
logger.warning("3 rows coerced to null in unit_price")
logger.error("could not reach the source API")

print()
print("pipeline.log now holds the same lines, with timestamps:")
with open("pipeline.log", encoding="utf-8") as f:
    print(f.read())

| Level | Use it for |
|---|---|
| `DEBUG` | Detail you want while developing, off in production |
| `INFO` | Normal progress. "extracted 40,132 rows" |
| `WARNING` | Something is odd but the run continues. Coerced values, a retry |
| `ERROR` | This step failed |
| `CRITICAL` | The pipeline cannot continue at all |

The rule of thumb: log every **count** at INFO. Row counts in and out of every step
are what let you debug a run you were not watching.

In [ ]:
# Config in one place, not scattered through the code
CONFIG = {
    "pipeline": "orders_daily",
    "sources": {
        "files": {"pattern": "landing/orders_*.csv"},
        "database": {"path": "source.db", "table": "customers"},
        "api": {"url": f"{API_URL}/api/products"},
    },
    "target": {"path": "warehouse.db", "table": "fact_orders"},
    "quality": {"max_reject_rate": 0.40, "min_rows": 1},
    "retries": {"attempts": 3, "base_delay": 0.2},
}

import json
with open("pipeline_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

print(json.dumps(CONFIG, indent=2)[:400], "...")

Config in a file means you can change the reject threshold or point at a different
source without editing, testing and redeploying code. It also means the config can
differ between your laptop and production without a single `if` statement.

In [ ]:
# A retry decorator, so any step can be made resilient without repeating yourself
import time, functools


def with_retry(attempts=3, base_delay=0.2, exceptions=(Exception,)):
    """Retry the wrapped function with exponential backoff."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            delay = base_delay
            for attempt in range(1, attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == attempts:
                        logger.error(f"{func.__name__} failed after {attempts} attempts")
                        raise
                    logger.warning(
                        f"{func.__name__} attempt {attempt} failed "
                        f"({type(e).__name__}), retrying in {delay:.1f}s")
                    time.sleep(delay)
                    delay *= 2
        return wrapper
    return decorator


_calls = {"n": 0}


@with_retry(attempts=4, base_delay=0.1)
def flaky_step():
    """Pretend this is a network call that fails twice."""
    _calls["n"] += 1
    if _calls["n"] < 3:
        raise ConnectionError("source unreachable")
    return f"succeeded on attempt {_calls['n']}"


print(flaky_step())

A decorator is a function that wraps another function. `@with_retry(...)` above a
definition means "replace this function with the retrying version of itself". You now
get retries on any step by adding one line, instead of copying a loop into five
places.

In [ ]:
# Recording every run, so you can answer "when did this last work?"
def log_run(pipeline, started, finished, read, loaded, rejected_count,
            status, message, db="warehouse.db"):
    """Write one row to etl_runs."""
    with closing(sqlite3.connect(db)) as wh:
        with wh:
            wh.execute(
                '''INSERT INTO etl_runs
                   (pipeline, started_at, finished_at, rows_read, rows_loaded,
                    rows_rejected, status, message)
                   VALUES (?, ?, ?, ?, ?, ?, ?, ?)''',
                (pipeline, started, finished, read, loaded, rejected_count,
                 status, message))


def show_runs(db="warehouse.db"):
    with closing(sqlite3.connect(db)) as wh:
        rows = wh.execute("SELECT * FROM etl_runs ORDER BY run_id").fetchall()
    print(f"{'id':<4}{'status':<10}{'read':<6}{'loaded':<8}{'rejected':<10}message")
    print("-" * 72)
    for row in rows:
        print(f"{row[0]:<4}{row[7]:<10}{row[4]:<6}{row[5]:<8}{row[6]:<10}{row[8]}")


log_run(CONFIG["pipeline"], datetime.now().isoformat(timespec="seconds"),
        datetime.now().isoformat(timespec="seconds"),
        len(raw_orders), len(enriched), len(rejected), "success", "first run")
show_runs()

In [ ]:
# Backfilling: the same job, run once per day of history
def run_for_date(run_date, dry_run=True):
    """Run the pipeline for one date. In a real job this filters the source."""
    logger.info(f"running for {run_date}{' (dry run)' if dry_run else ''}")
    path = f"landing/orders_{run_date}.csv"
    if not os.path.exists(path):
        logger.warning(f"no file for {run_date}, skipping")
        return 0
    rows = len(pd.read_csv(path))
    logger.info(f"{run_date}: {rows} row(s)")
    return rows


total = 0
for run_date in ["2026-05-01", "2026-05-02", "2026-05-03", "2026-05-04"]:
    total += run_for_date(run_date)

print(f"\nbackfilled {total} row(s)")

Notice `run_for_date(date)` takes the date as a **parameter** rather than calling
`date.today()` inside. That one choice is what makes a backfill possible. A pipeline
that hardcodes "today" can only ever run today, and reprocessing last Tuesday means
editing the code.

Every scheduler, Airflow included, passes the logical date in for exactly this reason.

### Your turn: tasks

**Task 1.** Set up a logger that writes to both the screen and `pipeline.log`, with
timestamps, and log one message at each of INFO, WARNING and ERROR.

**Task 2.** Move the thresholds and paths from your earlier functions into a `CONFIG`
dictionary, save it as JSON, and read it back.

**Task 3.** Write a `with_retry` decorator and use it on a function that fails twice
before succeeding.

**DE Task 4.** Write `log_run(...)` and `show_runs()` against the `etl_runs` table,
and record three runs, one of which is a failure.

**DE Task 5.** Write `run_for_date(run_date)` that processes one day's file and
returns the row count, skipping dates with no file. Loop it over four dates to
backfill.

**DE Task 6.** Add a `dry_run=True` parameter to your pipeline that does all the
extraction, transformation and validation, prints the report, and writes nothing.

In [ ]:
# Task 1 answer


In [ ]:
# Task 2 answer


In [ ]:
# Task 3 answer


In [ ]:
# DE Task 4 answer


In [ ]:
# DE Task 5 answer


In [ ]:
# DE Task 6 answer


## 5. Scheduling Pipelines with Airflow
Airflow is a powerful tool for scheduling and monitoring data pipelines.

**Reference: an Airflow DAG (the old API, kept for comparison)**

This is the version that appeared here originally. It will not parse on Airflow 2 or
3, because `airflow.operators.python_operator` was removed and `schedule_interval`
was renamed. The corrected versions, in both the operator and TaskFlow styles, are in
the Airflow section below.

```python
from airflow import DAG
from airflow.operators.python_operator import PythonOperator   # moved
from datetime import datetime

def extract_task():
    pass

def transform_task():
    pass

def load_task():
    pass

default_args = {"owner": "airflow", "start_date": datetime(2023, 1, 1)}

dag = DAG("etl_pipeline", default_args=default_args, schedule_interval="@daily")  # renamed

extract = PythonOperator(task_id="extract", python_callable=extract_task, dag=dag)
transform = PythonOperator(task_id="transform", python_callable=transform_task, dag=dag)
load = PythonOperator(task_id="load", python_callable=load_task, dag=dag)

extract >> transform >> load
```

---

## Scheduling with Airflow

Airflow is not installed here, so this section is reference rather than runnable.
The DAG earlier in this notebook uses the old API, which stopped working in
Airflow 2.x and 3.x. Here is what changed.

### What moved

| Old | Current |
|---|---|
| `from airflow.operators.python_operator import PythonOperator` | `from airflow.providers.standard.operators.python import PythonOperator` (Airflow 3) or `from airflow.operators.python import PythonOperator` (Airflow 2) |
| `schedule_interval='@daily'` | `schedule='@daily'` |
| `start_date=datetime(2023, 1, 1)` with no `catchup` | always set `catchup` explicitly |
| `provide_context=True` | gone, context is passed automatically |
| `execution_date` | `logical_date`, or the `ds` template variable |

Leaving `catchup` unset with a `start_date` two years ago means Airflow will try to
run 700 backfill jobs the moment you unpause the DAG. Set it deliberately, every
time.

### The classic operator style

```python
from datetime import datetime
from airflow import DAG
from airflow.operators.python import PythonOperator      # airflow 2.x


def extract_task(**context):
    run_date = context["ds"]                # the logical date, as YYYY-MM-DD
    # ... extract for that date, return a path rather than a DataFrame
    return f"raw/date={run_date}/orders.csv"


def transform_task(**context):
    path = context["ti"].xcom_pull(task_ids="extract")
    # ... read, clean, write
    return path.replace("raw", "staged")


def load_task(**context):
    path = context["ti"].xcom_pull(task_ids="transform")
    # ... load into the warehouse


default_args = {
    "owner": "data-engineering",
    "retries": 3,
    "retry_delay": timedelta(minutes=5),
    "email_on_failure": True,
}

with DAG(
    dag_id="orders_daily",
    default_args=default_args,
    start_date=datetime(2026, 1, 1),
    schedule="@daily",
    catchup=False,
    max_active_runs=1,
    tags=["orders", "dec"],
) as dag:

    extract = PythonOperator(task_id="extract", python_callable=extract_task)
    transform = PythonOperator(task_id="transform", python_callable=transform_task)
    load = PythonOperator(task_id="load", python_callable=load_task)

    extract >> transform >> load
```

### The TaskFlow style, which is what new DAGs use

```python
from datetime import datetime, timedelta
from airflow.decorators import dag, task


@dag(
    start_date=datetime(2026, 1, 1),
    schedule="@daily",
    catchup=False,
    default_args={"retries": 3, "retry_delay": timedelta(minutes=5)},
    tags=["orders", "dec"],
)
def orders_daily():

    @task
    def extract(ds=None):
        return f"raw/date={ds}/orders.csv"

    @task
    def transform(path: str):
        return path.replace("raw", "staged")

    @task
    def load(path: str):
        print(f"loading {path}")

    load(transform(extract()))


orders_daily()
```

Same pipeline, no `xcom_pull`, and the dependencies come from how you call the
functions. Values passed between tasks still go through XCom underneath.

### Three rules that matter more than the syntax

**1. Pass data by reference, not by value.** XCom is a small metadata table, not a
data store. Return a file path or a partition name from a task, never a DataFrame.
A 500MB DataFrame through XCom will take down your metadata database.

**2. Tasks must be idempotent.** Airflow retries failed tasks and reruns backfills by
design. A task that inserts rather than upserts will duplicate data on the first
retry, which is a matter of when, not if.

**3. Use the logical date, never `today()`.** `ds` is the date the run represents.
A backfill of last March passes `2026-03-14`, and your task must honour it. Calling
`date.today()` inside a task means a backfill silently reprocesses today's data
thirty times.

Everything in this notebook has been building towards those three, which is why the
functions here take a date parameter and the loads are all upserts.

### Other schedulers you will meet

| Tool | Worth knowing |
|---|---|
| **Airflow** | The default in most companies. Python DAGs, huge ecosystem |
| **Dagster** | Asset-centric: you declare the tables, it works out the order |
| **Prefect** | Lighter, plain Python functions, less infrastructure |
| **dbt** | Transformation only, in SQL, inside the warehouse. Usually orchestrated by one of the above |
| **cron** | Still perfectly good for one script on one machine. Not everything needs a DAG |

The concepts move between them unchanged: a DAG of tasks, retries, a schedule,
idempotency and a logical date.

---

## End of class mini project: the whole pipeline

Three sources, one warehouse, safe to run any number of times. Everything you need is
already in this notebook.

**Sources:** daily CSVs in `landing/`, the `customers` table in `source.db`, and the
products API at `API_URL`.

**Target:** `warehouse.db`, with `fact_orders`, `dim_customers`, `dim_products`,
`rejected_orders` and `etl_runs`.

### What to build

Write these functions, each doing one job, each logging its counts.

**Extract**
1. `extract_files(pattern)` reads the daily CSVs with `dtype=str`, tagging the source
   file.
2. `extract_customers(since=None)` reads `source.db`, incrementally when given a
   watermark.
3. `extract_products(url)` pages through the API.
4. Land all three under `raw/date=<run_date>/` and write a `_manifest.json`.

**Transform**
5. `standardise(df)`, `coerce_types(df)`, `deduplicate(df, key)` as built above.
6. `validate(df, known_customers)` returning `(clean, rejected)` with reasons,
   including the referential integrity check.
7. `enrich(orders, customers, products)` joining both dimensions (`validate=
   "many_to_one"` on each) and adding `revenue`, `vat`, `total`, `margin`
   (revenue minus quantity times product cost), `order_date` and `value_band`.

**Quality**
8. `quality_gates(...)` with at least five checks. At minimum: rejection rate under
   40%, at least one clean row, no duplicate `order_id`, no negative revenue, and
   every `order_date` present.

**Load**
9. `load_dim_customers(df)` and `load_dim_products(df)`, both upserts.
10. `load_fact(df)` upserting on `order_id`.
11. `load_rejects(rejected)`.
12. `verify_load(df)` comparing expected against actual row count and revenue.

**Orchestrate**
13. `run(run_date=None, dry_run=False)` that calls everything in order, logs to both
    the screen and `pipeline.log`, writes a row to `etl_runs` whether it succeeds or
    fails, and returns a summary dict.

### Prove it works

- Run it three times. `fact_orders` must have the same row count every time.
- Change a status in one landing file, re-run, and show the warehouse row was
  updated rather than duplicated.
- Lower the reject threshold to 0.1 and show the gate stopping the load, with a
  `failed` row appearing in `etl_runs`.
- Print `etl_runs` at the end. Every run accounted for.

### Stretch goals

14. Backfill: loop `run(run_date=d)` over the three dates and confirm the totals.
15. Add `margin_pct` and a `dim_products` join that flags orders for items the
    product API has never heard of.
16. Write the fact table out as partitioned files as well, and reconcile the file
    totals against the database.
17. Add a `--since` style incremental mode for customers using `etl_watermark`,
    and show the second run extracting zero customer rows.

In [ ]:
# Extract



In [ ]:
# Transform



In [ ]:
# Quality gates



In [ ]:
# Load



In [ ]:
# run()



In [ ]:
# prove it: three runs, then the run log



---

## Common mistakes to watch for

| Mistake | What you see | The fix |
|---|---|---|
| Cleaning during extract | No way to explain a number six months later | Land raw, transform a copy |
| Letting pandas infer dtypes at extract | Column type changes between runs | `dtype=str`, convert in transform |
| `errors="coerce"` with no count | Silent data loss | Report how many values it swallowed |
| Bare `drop_duplicates()` | Near-duplicates survive | `subset=["business_key"]` |
| No referential integrity check | Orphan facts, broken joins downstream | Check keys against the dimension |
| Deriving before validating | Wasted work on rows you throw away | Validate first |
| Checks with no consequence | A red number nobody acts on | Make it a gate that stops the load |
| Plain `INSERT` in the load | Duplicates on the first retry | `ON CONFLICT DO UPDATE` |
| Not verifying after load | You hope it worked | Compare counts and totals |
| Dropping bad rows silently | Data vanishes, nobody knows | A rejects table with reasons |
| Merge without `validate=` | Revenue quietly doubles | `validate="many_to_one"` |
| `print` instead of logging | Nothing to read after a 3am failure | `logging`, to a file, with counts |
| `date.today()` inside the job | Backfills reprocess today, repeatedly | Pass `run_date` as a parameter |
| No run log | "When did this last work?" is unanswerable | An `etl_runs` table |
| DataFrames through XCom | The Airflow metadata DB falls over | Pass paths, not data |
| `schedule_interval` and old imports | The DAG will not parse on Airflow 2 or 3 | `schedule=`, current import paths |
| `catchup` left unset | 700 backfill runs start at once | Set it explicitly |
| Watermark with `>=` | The boundary row reloads every run | Use `>` |
| Watermark on `created_at` | Late-arriving updates are missed | Use `updated_at` |

### The checklist for any pipeline you build

Before you call a pipeline finished, it should be able to answer yes to all of these:

1. Can I run it twice without creating duplicates?
2. Can I run it for a date in the past?
3. If it fails halfway, is the target still consistent?
4. Does it log how many rows went in and how many came out, at every step?
5. If a row is dropped, can someone find out which one and why?
6. Will it stop rather than load, when the data is clearly wrong?
7. Does it verify its own output?
8. Is there a record of every run, including the failures?
9. Are the credentials outside the code?
10. Could a colleague re-run it tomorrow without asking me anything?

Your notebooks so far have covered every one of these. The capstone above is where
they all meet.

In [ ]:
# Clean up everything this notebook created.
# Run this only when you are finished.
import os, shutil

for name in ["source.db", "warehouse.db", "example.db", "example.csv",
             "transformed_data.csv", "final_output.csv", "pipeline.log",
             "pipeline_config.json"]:
    if os.path.isfile(name):
        os.remove(name)

for folder in ["landing", "raw", "warehouse_files"]:
    if os.path.isdir(folder):
        shutil.rmtree(folder)

try:
    _api.shutdown()
    print("practice API stopped")
except Exception:
    pass

print("cleaned up. re-run the setup cell at the top to start over")